In [98]:
from langchain_ollama import ChatOllama, OllamaLLM

MODEL_NAME = "qwen2.5:3b"
SAVE_FILE_NAME = "generator_eval_qwen2.5_3b.csv"

chat_model = ChatOllama(model=MODEL_NAME)

model = OllamaLLM(model=MODEL_NAME)

In [99]:
model.invoke("Say hello")

'Hello! How can I assist you today?'

In [100]:
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import PromptTemplate, ChatPromptTemplate

template = """
You are a helpful and friendly Next.js assistant. 
Your responsibility is to answer user queries about Next.js. 
Answer the question based only and only on the given context below. If you can't answer the question, reply "I don't know".

Context: {context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)

chat_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", 
         """
You are a helpful and friendly Next.js assistant. 
Your responsibility is to answer user queries about Next.js. 
Answer the question based only and only on the below context (which got from Next.js documentation). If you can't answer the question, reply \"I don't know\".

Context: {context}

Question: {question}"""),
    ]
)

parser = StrOutputParser()

chain = prompt | model | parser
chat_chain = chat_prompt | chat_model | parser

In [101]:
# Parse code content to JSON
import json
import re

change_json_pattern = r"\'[^\'\"]*?\'\: \'[^\'\"]*?\'"
change_code_json_pattern = r"\"code\"\: \'[^\'\"]*?\',"

def json_reformat(code: str):
    for match in re.finditer(change_json_pattern, code):
        found = match.group().replace("'", '"')
        code = code.replace(match.group(), found)
        code = code.replace("'code': ", '"code": ')
        code = code.replace("'switcher': ", '"switcher": ')
    for match in re.finditer(change_code_json_pattern, code):
        found = match.group().replace("'", '"')
        code = code.replace(match.group(), found)
    return code

def parse_code_content(code_content: str):
    code_content = json_reformat(code_content)
    # write to temp.json
    code_content = code_content.replace("\n", "\\n")
    code_content = code_content.replace("True", "true")
    code_content = code_content.replace("False", "false")
    with open("temp.json", "w") as f:
        f.write(code_content)
    code_content_json = json.loads(code_content)
    return code_content_json

def place_snippets_in_text(text_content: str, code_content_json: list):
    # return text_content + code_content_json
    code_template = "code_snippet_"
    text_content_add_snippets = text_content
    for i in range(len(code_content_json)):
        code_location = code_template + str(i+1)
        code_snippet = ""
        if code_content_json[i]['language']:
            code_snippet += code_content_json[i]['language']
        if code_content_json[i]['filename']:
            code_snippet += f" filename=\"{code_content_json[i]['filename']}\""
        if code_content_json[i]['switcher']:
            switcher = code_content_json[i]['switcher']
            if switcher:
                code_snippet += " switcher"
        code_snippet += "\n"
        code_snippet += code_content_json[i]['code']
        text_content_add_snippets = text_content_add_snippets.replace(code_location, code_snippet)
    return text_content_add_snippets



In [102]:
import pandas
pandas.set_option('display.max_colwidth', None)

df = pandas.read_csv("../data/processed_entries_v15.1.3_batch_2.csv")

In [103]:
df['text_content'][6]

"Dynamic Segments can be extended to catch-all subsequent segments by adding an ellipsis inside the brackets [...folderName].\nFor example, app/shop/[...slug]/page.js will match /shop/clothes, but also /shop/clothes/tops, /shop/clothes/tops/t-shirts, and so on.\n| Route | Example URL | params |\n| :--- | :--- | :--- |\n| app/shop/[...slug]/page.js | /shop/a | { slug: ['a'] } |\n| app/shop/[...slug]/page.js | /shop/a/b | { slug: ['a', 'b'] } |\n| app/shop/[...slug]/page.js | /shop/a/b/c | { slug: ['a', 'b', 'c'] } |\n"

In [104]:
list_of_chunks = [df.loc[3], df.loc[5]]

retrieved_data = ""

for chunk in list_of_chunks:
    # print(chunk['code_content'])
    code_content_json = parse_code_content(chunk['code_content'])
    original_chunk = place_snippets_in_text(chunk['text_content'], code_content_json)
    retrieved_data += original_chunk
    retrieved_data += "\n\n"

# print(retrieved_data)

In [105]:
answer = chain.invoke({"context": retrieved_data, "question": "How to setup dynamic routes in Next.js?"})

In [106]:
print(answer)

To set up dynamic routes in Next.js, you can follow these steps:

1. **Define Dynamic Route Segment**: In your application’s routing file (usually `.js` or `.ts` files), you define the route with a dynamic segment like this:
   ```tsx
   import { lazy } from 'react'
   import dynamic from 'next/dynamic'

   const LazyComponent = dynamic(lazy(() => import('./components/YourComponent')), {
     ssr: false,
   })

   export default function YourRoute() {
     return (
       <div>
         {/* Route to a component based on the slug */}
         <LazyComponent />
       </div>
     )
   }

   // In your app directory, you might have something like this:
   import dynamic from 'next/dynamic'
   import { generateStaticParams } from './components/YourComponent'

   export default function YourRoute() {
     return (
       <>
         {/* Routes to a component based on the slug */}
         <dynamic {...generateStaticParams()} />
       </>
     )
   }
   ```

2. **Create Page Files**: Create

In [107]:
data_frame = pandas.DataFrame({"entry_ids": "3 5", "context": [retrieved_data], "question": ["How to setup dynamic routes in Next.js?"], "answer": answer})

data_frame

,entry_ids,context,question,answer
0,3 5,"For example, a blog could include the following route app/blog/[slug]/page.js where [slug] is the Dynamic Segment for blog posts.\n```tsx filename=""app/blog/[slug]/page.tsx"" switcher\nexport default async function Page({\n params,\n}: {\n params: Promise<{ slug: string }>\n}) {\n const slug = (await params).slug\n return <div>My Post: {slug}</div>\n}\n```\n```jsx filename=""app/blog/[slug]/page.js"" switcher\nexport default async function Page({ params }) {\n const slug = (await params).slug\n return <div>My Post: {slug}</div>\n}\n```\n| Route | Example URL | params |\n| :--- | :--- | :--- |\n| app/blog/[slug]/page.js | /blog/a | { slug: 'a' } |\n| app/blog/[slug]/page.js | /blog/b | { slug: 'b' } |\n| app/blog/[slug]/page.js | /blog/c | { slug: 'c' } |\nSee the generateStaticParams() page to learn how to generate the params for the segment.\n\n\nThe generateStaticParams function can be used in combination with dynamic route segments to statically generate routes at build time instead of on-demand at request time.\n```tsx filename=""app/blog/[slug]/page.tsx"" switcher\nexport async function generateStaticParams() {\n const posts = await fetch('https://.../posts').then((res) => res.json())\n\n return posts.map((post) => ({\n slug: post.slug,\n }))\n}\n```\n```jsx filename=""app/blog/[slug]/page.js"" switcher\nexport async function generateStaticParams() {\n const posts = await fetch('https://.../posts').then((res) => res.json())\n\n return posts.map((post) => ({\n slug: post.slug,\n }))\n}\n```\nThe primary benefit of the generateStaticParams function is its smart retrieval of data. If content is fetched within the generateStaticParams function using a fetch request, the requests are automatically memoized. This means a fetch request with the same arguments across multiple generateStaticParams, Layouts, and Pages will only be made once, which decreases build times.\nUse the migration guide if you are migrating from the pages directory.\nSee generateStaticParams server function documentation for more information and advanced use cases.\n\n\n",How to setup dynamic routes in Next.js?,"To set up dynamic routes in Next.js, you can follow these steps:\n\n1. **Define Dynamic Route Segment**: In your application’s routing file (usually `.js` or `.ts` files), you define the route with a dynamic segment like this:\n ```tsx\n import { lazy } from 'react'\n import dynamic from 'next/dynamic'\n\n const LazyComponent = dynamic(lazy(() => import('./components/YourComponent')), {\n ssr: false,\n })\n\n export default function YourRoute() {\n return (\n <div>\n {/* Route to a component based on the slug */}\n <LazyComponent />\n </div>\n )\n }\n\n // In your app directory, you might have something like this:\n import dynamic from 'next/dynamic'\n import { generateStaticParams } from './components/YourComponent'\n\n export default function YourRoute() {\n return (\n <>\n {/* Routes to a component based on the slug */}\n <dynamic {...generateStaticParams()} />\n </>\n )\n }\n ```\n\n2. **Create Page Files**: Create page files for each route, even if it's just an empty file. For example:\n ```tsx\n // app/blog/[slug]/page.tsx or app/blog/[slug].tsx\n export default function YourPage({ params }: { params: string[] }) {\n const { slug } = params\n\n return <div>Post Slug: {slug}</div>\n }\n ```\n\n3. **Fetch Data**: For dynamic routes, you can use `generateStaticParams` to fetch and statically generate the data needed for each route segment at build time or when server-side rendering (SSR) is enabled.\n ```tsx\n // In your component file (like app/blog/[slug]/page.tsx)\n import { generateStaticParams } from './components/YourComponent'\n\n export async function generateStaticParams() {\n const posts = await fetch('https://.../posts').then((res) => res.json())\n\n return posts.map((post) => ({\n slug: post.slug,\n }))\n }\n ```\n\n4. **Use Switch**: The `Switch` component is useful when you have multiple dy

In [108]:
data_frame.to_csv(SAVE_FILE_NAME, index=False)

In [109]:
# idx = "3 5"

# con = """For example, a blog could include the following route app/blog/[slug]/page.js where [slug] is the Dynamic Segment for blog posts.\n```tsx filename="app/blog/[slug]/page.tsx" switcher\nexport default async function Page({\n params,\n}: {\n params: Promise<{ slug: string }>\n}) {\n const slug = (await params).slug\n return <div>My Post: {slug}</div>\n}\n```\n```jsx filename="app/blog/[slug]/page.js" switcher\nexport default async function Page({ params }) {\n const slug = (await params).slug\n return <div>My Post: {slug}</div>\n}\n```\n| Route | Example URL | params |\n| :--- | :--- | :--- |\n| app/blog/[slug]/page.js | /blog/a | { slug: 'a' } |\n| app/blog/[slug]/page.js | /blog/b | { slug: 'b' } |\n| app/blog/[slug]/page.js | /blog/c | { slug: 'c' } |\nSee the generateStaticParams() page to learn how to generate the params for the segment.\n\n\nThe generateStaticParams function can be used in combination with dynamic route segments to statically generate routes at build time instead of on-demand at request time.\n```tsx filename="app/blog/[slug]/page.tsx" switcher\nexport async function generateStaticParams() {\n const posts = await fetch('https://.../posts').then((res) => res.json())\n\n return posts.map((post) => ({\n slug: post.slug,\n }))\n}\n```\n```jsx filename="app/blog/[slug]/page.js" switcher\nexport async function generateStaticParams() {\n const posts = await fetch('https://.../posts').then((res) => res.json())\n\n return posts.map((post) => ({\n slug: post.slug,\n }))\n}\n```\nThe primary benefit of the generateStaticParams function is its smart retrieval of data. If content is fetched within the generateStaticParams function using a fetch request, the requests are automatically memoized. This means a fetch request with the same arguments across multiple generateStaticParams, Layouts, and Pages will only be made once, which decreases build times.\nUse the migration guide if you are migrating from the pages directory.\nSee generateStaticParams server function documentation for more information and advanced use cases.\n\n\n	"""

# que = "How to setup dynamic routes in Next.js?"

# ans = """To set up dynamic routes in Next.js, you can follow these steps:\n\n1. Create a new file called `next.config.js` in the root directory of your Next.js project.\n\n```javascript\nmodule.exports = {\n // Set the base URL for your app\n baseURL: process.env.BASE_URL,\n // Define a route that maps to a specific page in your app\n routes: [\n { path: '/page', component: PageComponent } ],\n // Enable dynamic routing by default in your Next.js project\n dynamicRoutesEnabledByDefault: true\n};\n```\n\n2. In the `next.config.js` file, create an object called `pages` that maps to a specific page in your app.\n\n```javascript\nconst pages = {\n '/page1': PageComponent1,\n '/page2': PageComponent2,\n '/page3': PageComponent3,\n '/page4': PageComponent4,\n};\n\nmodule.exports = {\n baseURL: process.env.BASE_URL,\n routes: pages,\n dynamicRoutesEnabledByDefault: true\n};\n```\n\n3. In the `PageComponent.js` file, create a basic component that displays information about a specific page in your app.\n\n```javascript\nexport default class PageComponent extends React.Component {\n constructor(props) {\n super(props);\n this.state = { slug: props.params.slug } };\n render() {\n return (\n <div className="page container">\n <h2>{this.props.children.props.title}}</h2>\n <p>{this.props.children.props.description}</p>\n <button onClick={() => this.setState({ slug: props.params.slug }) }}>Edit</button>\n <p>{this.props.children.props.readCount} times read</p>\n </div>\n );\n }\n}\n```\n\n4. In the `App.js` file, import the `next.config.js` file and use its dynamic routes functionality.\n\n```javascript\nimport { NextConfig } } from 'next'\n\nconst withDynamicRoutes = config => {\n config.plugins.push({\n name: 'dynamicroutes',\n async resolve() {\n // Check if a page slug is provided as the dynamic route prop.\n const pageSlugProp = 'page';\n \n // Fetch a specific page object in response to the page slug prop.\n const pageObject = await fetch(`/pages/${pageSlugProp}}`).then((res) => res.json())['data'] || undefined;\n \n // Check if the specific page object is available in response to the page slug prop.\n if (pageObject) {\n // Generate a dynamic route component based on the provided page slug prop, and return it as the output of the `withDynamicRoutes` function.\n const dynamicRouteComponent = (\n <div className="dynamic-route-container">\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n </div>\n )\n // Check if the specific page object is available in response to the page slug prop.\n else if (pageObject)) {\n // Generate a dynamic route component based on the provided page slug prop and return it as the output of the `withDynamicRoutes` function.\n const dynamicRouteComponent = (\n <div className="dynamic-route-container">\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n </div>\n )\n // Check if the specific page object is available in response to the page slug prop.\n else if (pageObject)) {\n // Generate a dynamic route component based on the provided page slug prop and return it as the output of the `withDynamicRoutes` function.\n const dynamicRouteComponent = (\n <div className="dynamic-route-container">\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n </div>\n )\n // Check if the specific page object is available in response to the page slug prop.\n else if (pageObject)) {\n // Generate a dynamic route component based on the provided page slug prop and return it as the output of the `withDynamicRoutes` function.\n const dynamicRouteComponent = (\n <div className="dynamic-route-container">\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n {/* Render the dynamic page component based on the provided page slug prop */}\n </div>\n )\n // Check if the specific page object is available in response to the page slug prop.\n else if (pageObject)) {\n // Generate a dynamic route component based on the provided page slug prop and return it as the output of the `withDynamicRoutes` function.\n const dynamicRouteComponent = (\n <div className="dynamic-route-container">\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n {/* Render the dynamic page component based on the provided page slug prop */}\n </div>\n )\n // Check if the specific page object is available in response to the page slug prop.\n else if (pageObject)) {\n // Generate a dynamic route component based on the provided page slug prop and return it as the output of the `withDynamicRoutes` function.\n const dynamicRouteComponent = (\n <div className="dynamic-route-container">\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n {/* Render the dynamic page component based on the provided page slug prop */}\n </div>\n )\n // Check if the specific page object is available in response to the page slug prop.\n else if (pageObject)) {\n // Generate a dynamic route component based on the provided page slug prop and return it as the output of the `withDynamicRoutes` function.\n const dynamicRouteComponent = (\n <div className="dynamic-route-container">\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n {/* Render the dynamic page component based on the provided page slug prop */}\n </div>\n )\n // Check if the specific page object is available in response to the page slug prop.\n else if (pageObject)) {\n // Generate a dynamic route component based on the provided page slug prop and return it as the output of the `withDynamicRoutes` function.\n const dynamicRouteComponent = (\n <div className="dynamic-route-container">\n {/* Render the dynamic page component based on the provided page slug prop */}\n\n {/* Render the dynamic page component based on the provided page slug prop */}\n </div>\n )\n // Check if the specific page object is available in response to the page slug prop.\n else if (pageObject)) {\n // Generate a dynamic route component based on the provided page slug prop and return it as the output of the `withDynamicRoutes` function.\n const dynamicRouteComponent = (\n <div className="dynamic-route-container">\n {/* Render the dynamic page component based on the provided page slug prop */}\n </div>\n )\n // Check if the specific page object is available in response to the page slug prop.\n else if (pageObject)) {\n // Generate a dynamic route component based on the provided page slug prop and return it as the output of the `withDynamicRoutes` function.\n const dynamicRouteComponent = (\n <div className="dynamic-route-container">\n {/* Render the dynamic page component based on the provided page slug prop */}\n </div>\n )\n // Check if the specific page object is available in response to the page slug prop.\n else if (pageObject)) {\n // Generate a dynamic route component based on the provided page slug prop and return it as the output of the `withDynamicRoutes` function.\n const dynamicRouteComponent = (\n <div className="dynamic-route-container">\n {/* Render the dynamic page component based on the provided page slug prop */}\n </div>\n )\n // Check if the specific page object is available in response to the page slug prop.\n else if (pageObject)) {\n // Generate a dynamic route component based on the provided page slug prop and return it as the output of the `withDynamicRoutes` function.\n const dynamicRouteComponent = (\n <div className="dynamic-route-container">\n {/* Render the dynamic page component based on the provided page slug prop */}\n </div>\n )\n // Check if the specific page object is available in response to the page slug prop.\n else if (pageObject)) {\n // Generate a dynamic route component based on the provided page slug prop and return it as the output of the `withDynamicRoutes` function.\n const dynamicRouteComponent = (\n <div className="dynamic-route-container">\n {/* Render the dynamic page component based on the provided page slug prop */}\n </div>\n )\n }\n // Return the dynamically generated route component\n return dynamicRouteComponent;\n }\n}\n```\n\nIn this example, we're using a Next.js app to generate the dynamic route components. The `dynamicRouteComponent` variable is returned by the `withDynamicRoutes` function that generates the component based on the provided page slug prop.\nNote: In order for this code to work with Next.js, you'll need to set up your server and configure it to use Next.js. This process typically involves installing Next.js and setting up any necessary API keys or OAuth tokens.\nFor more detailed information about how to set up a Next.js app and configure it to use the dynamic route components generated by this code, please refer to the official Next.js documentation. Additionally, you may find helpful resources such as tutorials, guides, and forums dedicated to Next.js, helping you learn and get started with Next.js.\n번역결과\n\n이 코드는 Next.js 앹에 생성되는 디자인 레이어-component를 반환하는 데 사용됩니다.\nNext.js는 서버와 인터페이스를 통해 편리하게 구현할 수 있는 JavaScript库입니다. 다음으로, Next.js 앹의 편리 구현에 대한 다음 설명을 시작합니다.\n\n이 코드는 Next.js 앹에 생성되는 디자인 레이어-component을 반환하는 데 사용됩니다.\nNext.js는 서버와 인터페이스를 통해 편리하게 구현할 수 있는 JavaScript库입니다. 다음으로, Next.js 앹의 편리 구현에 대한 다음 설명을 시작합니다.\n\n이 코드는 Next.js 앹에 생성되는 디자인 레이어-component를 반환하는 데 사용됩니다.\nNext.js는 서버와 인터페이스를 통해 편리하게구현할 수 있는 JavaScript库입니다. 다음으로, Next.js 앹의 편리 구현에 대한 다음 설명을 시작니다.\n\n이 코드는 Next.js 앹에 생성되는 디자인 레이어-component를 반환하는 데 사용됩니다.\nNext.js는 서버와 인터페이스를 통해 편리하게구현할 수 있는 JavaScript库입니다. 다음으로, Next.js 액의 편리 구현에 대한 다음 설명을 시작니다.\n\n이 코드는 Next.js 액에 생성되는 디자인 레이어-component를 반환하는 데 사용됩니다.\nNext.js는 서버와 인터페이스를 통해 편리하게구현할 수 있는 JavaScript库입니다. 다음으로, Next.js 액의 편리 구현에 대한 다음 설명을 시작니다.\n\n이 코드는 Next.js 액에 생성되는 디자인 레이어-component를 반환하는 데 사용됩니다.\nNext.js는 서버와 인터페이스를 통해 편리하게구현할 수 있는 JavaScript库입니다. 다음으로, Next.js 액의 편리 구현에 대한 다음 설명을 시작니다.\n\n이 코드는 Next.js 액에 생성되는 디자인 레이어-component를 반환하는 데 사용됩니다.\nNext.js는 서버와 인터페이스를 통해 편리하게구현할 수 있는 JavaScript库입니다. 다음으로, Next.js 액의 편리 구현에 대한 다음 설명을 시작니다.\n\n이 코드는 Next.js 액에 생성되는 디자인 레이어-component를 반환하는 데 사용됩니다.\nNext.js는 서버와 인터페이스를 통해 편리하게구현할 수 있는 JavaScript库입니다. 다음으로, Next.js 액의 편리 구현에 대한 다음 설명을 시작니다.\n\n이 코드는 Next.js 액에 생성되는 디자인 레이어-component를 반환하는 데 사용됩니다.\nNext.js는 서버와 인터페이스를 통해 편리하게구현할 수 있는 JavaScript库입니다. 다음과 같은 예제를 제공하여 다음 설명을 시작하겠습니다.\n\n```jsx\nimport { render, fireEvent } from '@testing-library/react';\n\n// 创建一个简单的按钮组件\nclass SimpleButton extends React.Component {\n constructor(props) {\n super(props);\n this.state = { clicked: false } };\n }\n\n handleClick() {\n this.setState({ clicked: true }) });\n }\n\n render() {\n return (\n <button onClick={this.handleClick.bind(this)}} disabled="disabled">Simple Button</button>\n );\n }\n}\n\n// 创建一个简单的按钮组件并渲染\nconst buttonComponent = (\n<div className="container">\n<button onClick={() => { this.setState({ clicked: true }) }} disabled="disabled">Simple Button</button>\n</div>\n));\n\n// 创建一个简单的测试用例\ntest('renders a simple button component', () => {\n // Create a new SimpleButton component\n const simpleButtonComponent = (\n<div className="container">\n<button onClick={() => { this.setState({ clicked: true }) }} disabled="disabled">Simple Button</button>\n</div>\n));\n\n// Render the test case and verify that it renders a simple button component\nrender(simpleButtonComponent), () => {\n // Verify that the component rendered has the state variable 'clicked' set to true\n expect(simpleButtonComponent.state('clicked')).toBe(true));\n});\n```\n\n위 코드를 실행하면 다음과 같이 라이브러리인 `Next.js` 앱의 `SimpleButton` 레이어를 만드는 JavaScript 프로그램을 생성합니다.\n다음은 생성된 JavaScript 프로그램입니다. 이 코드에서 SimpleButton 레이어를 생성하는 JavaScript 함수를 포함하고 있습니다.\n\n```jsx\nimport { render, fireEvent } from '@testing-library/react';\n\n// Create a new SimpleButton component\nconst simpleButtonComponent = (\n<div className="container">\n<button onClick={() => { this.setState({ clicked: true }) }} disabled="disabled">Simple Button</button>\n</div>\n));\n\n// Render the test case and verify that it renders a simple button component\nrender(simpleButtonComponent), () => {\n // Verify that the component rendered has the state variable 'clicked' set to true\n expect(simpleButtonComponent.state('clicked')).toBe(true));\n});\n```\n\n위 코드를 실행하면 다음과 같이 라이브러리인 `Next.js` 앱의 `SimpleButton` 레이어를 생성하는 JavaScript 함수를 포함하고 있습니다.\n\n```jsx\nimport { render, fireEvent } from '@testing-library/react';\n\n// Create a new SimpleButton component\nconst simpleButtonComponent = (\n<div className="container">\n<button onClick={() => { this.setState({ clicked: true }) }} disabled="disabled">Simple Button</button>\n</div>\n ));\n\n// Render the test case and verify that it renders a simple button component\nrender(simpleButtonComponent), () => {\n // Verify that the component rendered has the state variable 'clicked' set to true\n expect(simpleButtonComponent.state('clicked')).toBe(true));\n});\n```\n\n위 코드를 실행하면 다음과 같이 라이브러리인 `Next.js` 앱의 `SimpleButton` 레이어를 생성하는 JavaScript 함수를 포함하고 있습니다.\n\n```jsx\nimport { render, fireEvent } from '@testing-library/react';\n\n// Create a new SimpleButton component\nconst simpleButtonComponent = (\n<div className="container">\n<button onClick={() => { this.setState({ clicked: true }) }} disabled="disabled">Simple Button</button>\n</div>\n ));\n\n// Render the test case and verify that it renders a simple button component\nrender(simpleButtonComponent), () => {\n // Verify that the component rendered has the state variable 'clicked' set to true\n expect(simpleButtonComponent.state('clicked')).toBe(true));\n});\n```\n\n위 코드를 실행하면 다음과 같이 라이브러리인 `Next.js` 앱의 `SimpleButton` 레이어를 생성하는 JavaScript 함수를 포함하고 있습니다.\n\n```jsx\nimport { render, fireEvent } from '@testing-library/react';\n\n// Create a new SimpleButton component\nconst simpleButtonComponent = (\n<div className="container">\n<button onClick={() => { this.setState({ clicked: true }) }} disabled="disabled">Simple Button</button>\n</div>\n ));\n\n// Render the test case and verify that it renders a simple button component\nrender(simpleButtonComponent), () => {\n // Verify that the component rendered has the state variable 'clicked' set to true\n expect(simpleButtonComponent.state('clicked')).toBe(true));\n});\n```\n\n위 코드를 실행하면 다음과 같이 라이브러리인 `Next.js` 앱의 `SimpleButton` 레이어를 생성하는 JavaScript 함수를 포함하고 있습니다.\n\n```jsx\nimport { render, fireEvent } from '@testing-library/react';\n\n// Create a new SimpleButton component\nconst simpleButtonComponent = (\n<div className="container">\n<button onClick={() => { this.setState({ clicked: true }) }} disabled="disabled">Simple Button</button>\n</div>\n ));\n\n// Render the test case and verify that it renders a simple button component\nrender(simpleButtonComponent), () => {\n // Verify that the component rendered has the state variable 'clicked' set to true\n expect(simpleButtonComponent.state('clicked')).toBe(true));\n});\n```\n\n위 코드를 실행하면 다음과 같이 라이브러리인 `Next.js` 앱의 `SimpleButton` 레이어를 생성하는 JavaScript 함수를 포함하고 있습니다.\n\n```jsx\nimport { render, fireEvent } from '@testing-library/react';\n\n// Create a new SimpleButton component\nconst simpleButtonComponent = (\n<div className="container">\n<button onClick={() => { this.setState({ clicked: true }) }} disabled="disabled">Simple Button</button>\n</div>\n ));\n\n// Render the test case and verify that it renders a simple button component\nrender(simpleButtonComponent), () => {\n // Verify that the component rendered has the state variable 'clicked' set to true\n expect(simpleButtonComponent.state('clicked')).toBe(true));\n});\n```\n\n위 코드를 실행하면 다음과 같이 라이브러리인 `Next.js` 앱의 `SimpleButton` 레이어를 생성하는 JavaScript 함수를 포함하고 있습니다.\n\n```jsx\nimport { render, fireEvent } from '@testing-library/react';\n\n// Create a new SimpleButton component\nconst simpleButtonComponent = (\n<div className="container">\n<button onClick={() => { this.setState({ clicked: true }) }} disabled="disabled">Simple Button</button>\n</div>\n ));\n\n// Render the test case and verify that it renders a simple button component\nrender(simpleButtonComponent), () => {\n // Verify that the component rendered has the state variable 'clicked' set to true\n expect(simpleButtonComponent.state('clicked')).toBe(true));\n});\n```"""